SVG Generation with Qwen2.5-Coder Fine-tuning & Inference Pipeline

This repository contains the complete pipeline for fine-tuning a Large Language Model (Qwen2.5-Coder-1.5B) to generate valid, complex Scalable Vector Graphics (SVG) from natural language prompts.

The project is structured into four main phases
1. Data Pipeline: Cleaning noisy official data and synthesizing high-quality geometric compositions.
2. Model Training: Full-parameter equivalent fine-tuning using High-Rank LoRA on Google Colab (A100, T4 support).
3. Visual Debugging: An evolution gallery for visual checkpoint comparison.
4. Kaggle Inference: A robust batched inference pipeline designed for Kaggle's strict offline environments.

----------------------------------------------------------------------
Phase 1 Data Preparation
----------------------------------------------------------------------

Before training, you must prepare the final_train_v5.csv dataset locally.
This process purges low-quality paths from the official data and synthesizes
a pristine golden dataset to teach the model structural composition.

Prerequisites
Ensure the following three files are located in the same local directory
1. train.csv
   The official raw training data
2. purge_dataset_v5.py
   The data cleaning script
3. golden_generator.py
   The dynamic grammar and shape synthesis engine

Note: train.csv is not included in this repository due to GitHub file size limits.
Please download the official dataset from the competition page and place it in the project root directory.

Execution Steps

1. Run the Purge Script
This will filter out malformed XMLs, excessively complex paths, and normalize coordinate bounds.

Command
python purge_dataset_v5.py

Output
purged_train_v5.csv

2. Run the Golden Generator
This script will consume the purged data, append thousands of dynamically generated
compositional shapes, and shuffle the final dataset.

Command
python golden_generator.py

Output
final_train_v5.csv
This is your master training file.

----------------------------------------------------------------------
Phase 2: Training on Google Colab
----------------------------------------------------------------------

The training notebook (Colab_Train_Final.ipynb) is designed to run on Google Colab.
It features a robust dual-hardware configuration system, allowing it to run
efficiently on both entry-level (T4) and high-end (A100) GPUs.

1. Cloud Storage Setup

To prevent data loss and ensure fast IO during training, the notebook integrates
with Google Drive.

Steps:
- Open your Google Drive.
- At the root of your "My Drive", create a new folder named exactly:
  Kaggle_SVG
- Upload the final_train_v5.csv file (generated in Phase 1) into this
  Kaggle_SVG directory.

2. Hardware Configuration (T4 vs. A100)

*Note: The pre-trained results and weights provided in this submission were generated using an A100 GPU.*
*Expected Training Time: ~7 hours on a T4 GPU vs. ~1 hour on an A100 GPU.*

By default, the notebook is configured for the Free Tier T4 GPU to ensure accessibility and prevent Out-Of-Memory (OOM) errors:
- fp16 = True
- batch_size = 2
- optim = paged_adamw_8bit

If you have access to an A100 GPU (or a GPU with 50GB+ VRAM) and want to drastically reduce training time, you can modify the code to maximize throughput.

Look for the [A100 / T4 Switch Zone] comments in the following three cells and toggle the comments accordingly:

Cell 1 (Model Loading)
- Switch precision from float16 to bfloat16.

Cell 2 (Training)
- Increase per_device_train_batch_size to 8.
- Decrease gradient_accumulation_steps to 2.
- Switch the optimizer to adamw_torch_fused.
- Enable bf16=True (and comment out fp16=True).

Cell 3 (Model Merging)
- Ensure the loaded base model precision matches the precision used during
  training (switch from fp16 to bf16 for A100).

3. Execution

Once your hardware toggles are set, simply select "Run All" in Colab.
The notebook will automatically:
- Load the dataset
- Train the LoRA adapter
- Merge the LoRA weights back into the base model
- Export the final model to your Google Drive
----------------------------------------------------------------------
Phase 3 Visual Debugging (The Evolution Gallery)
----------------------------------------------------------------------

The final cell in the training notebook is the Evolution Gallery.
This is a powerful visual debugging tool.

Instead of guessing model performance by looking at loss curves, this tool
allows you to
- Input specific checkpoint step numbers saved during training
- Modify a custom text prompt
- Visually compare the SVG outputs generated by different checkpoints
  side-by-side

Why use this
It is highly effective for checking whether the model is underfitting or
overfitting, helping you pinpoint the absolute best epoch before committing
to Kaggle submission.

----------------------------------------------------------------------
Phase 4 Kaggle Inference
----------------------------------------------------------------------

To deploy your trained model in Kaggle's offline environment and generate
your final submission.csv

1. Model Export

After the Colab training finishes, locate the folder
merged_qwen_coder_1.5b_svg_final

Download the following 6 essential files
- chat_template.jinja
- config.json
- generation_config.json
- model.safetensors
- tokenizer_config.json
- tokenizer.json

2. Kaggle Deployment

Steps
- Create a new Dataset on Kaggle.
- Upload the 6 downloaded files to this new dataset.
- Open your Kaggle inference notebook (Kaggle_Inference_final.ipynb).
- Add Data Attach your newly uploaded model dataset to the notebook.
- Update the merged_model_path variable in the notebook so it points to
  your attached dataset directory, or let the auto-router find it.
- Click Run All.

The script will utilize dual T4 concurrency and smart fallback mechanisms
to process the test prompts and generate a resilient submission.csv.


In [ ]:
# ==========================================
# 🚀 Module 1: Global Initialization of Main Training Flow
# ==========================================
# 1. Install all dependencies at once
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets pandas==2.2.2 cairosvg

# 2. Unified import of all necessary training packages
import os
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

# 3. Mount cloud drive
print("🔗 Connecting to Google Drive...")
drive.mount('/content/drive')

# 4. Global core path configuration
data_path = '/content/drive/MyDrive/Kaggle_SVG/final_train_v5.csv'
base_model_id = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
output_dir = "/content/drive/MyDrive/Kaggle_SVG/results_full_train"
lora_save_path = "/content/drive/MyDrive/Kaggle_SVG/svg_lora_adapter_v5"

if os.path.exists(data_path):
    print(f"✅ Dataset successfully locked: {data_path}")
else:
    print(f"❌ Dataset not found, please check the path!")



In [ ]:
import random
import numpy as np
from transformers import set_seed

def seed_everything(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ['PYTHONHASHSEED'] = str(seed)

    set_seed(seed)

    print(f"Random Seeds Is: {seed}")


seed_everything(42)

In [ ]:
# Dataset Preparation

def format_prompt(examples):
    texts = [str(p) + str(s) for p, s in zip(examples["prompt"], examples["svg"])]
    return {"text": texts}


In [ ]:
# ==========================================
# 🚀 Module 2: Model Loading
# ==========================================

# Step 1: Loading of Base Model

print("🔄 Loading full-power base large model...")

# ------------------------------------------
# [ A100 Switch ] Native support for bfloat16. If you trained using A100 (bf16), Comment out the block below and uncomment this block:
# model = AutoModelForCausalLM.from_pretrained(
#     base_model_id,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
#     trust_remote_code=True
# )
# ------------------------------------------
# ⏬ [ T4 Switch ] T4 does not support bf16. If you trained using T4 (fp16), Comment out the block above and uncomment the block below:
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,   # Core change: Downgrade to fp16 to adapt for T4
    device_map="auto",
    trust_remote_code=True
)
# ------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)

# Check if the tokenizer fragments the tags
test_tokens = tokenizer.encode("<|im_start|>system")
print(f"🔍 Tokenizer fragmentation test IDs: {test_tokens}")
if len(test_tokens) > 3:
    print("⚠️ Warning: Tags are fragmented! This may degrade the model's instruction-following capability.")
else:
    print("✅ Safe: ChatML tags are intact!")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Core Protection: Take over gradient flow
model.gradient_checkpointing_enable()
model.enable_input_require_grads()


# Step 2: Inject Deluxe High-Rank LoRA

print("🔄 Injecting deluxe LoRA adapter...")
peft_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


In [ ]:
# ==========================================
# 🚀 Module 3: Dataset Loading and Formatting
# ==========================================
# Import dataset loading tool
from datasets import load_dataset

# Load your CSV file
dataset = load_dataset("csv", data_files=data_path)

# Extract the training set portion and assign it to train_dataset
train_dataset = dataset["train"]

# ==========================================
# Call the minimalist assembler and enable batch processing (batched=True)
# ==========================================
print("⏳ Assembling Prompt and SVG into a conversational format recognized by the model...")
train_dataset = train_dataset.map(format_prompt, batched=True, remove_columns=train_dataset.column_names)

print("✅ Dependencies installed, dataset successfully loaded and packaged!")
print(f"Number of records in the current dataset: {len(train_dataset)}")
print(f"Dataset column names (features): {train_dataset.column_names}") # Only ['text'] should remain here


In [ ]:
# ==========================================
# 🚀 Module 4: Training
# ==========================================
print("⚙️ Configuring training parameters...")

training_args = SFTConfig(
    output_dir=output_dir,

    # ================= A100 / T4 Switch Zone =================

    # per_device_train_batch_size=8,       # [A100] High concurrency throughput
    per_device_train_batch_size=2,     # [ T4 ] Prevents OOM (Comment out the A100 line above, uncomment this)

    # gradient_accumulation_steps=2,       # [A100] Combined with Batch Size for an effective batch of 16
    gradient_accumulation_steps=8,     # [ T4 ] Accumulates to effective batch 16 (Comment out the A100 line above, uncomment this)

    num_train_epochs=3,

    # bf16=True,                           # [A100] Enable native A100 acceleration
    fp16=True,                         # [ T4 ] T4 specific precision (Comment out the A100 line above, uncomment this)

    # optim="adamw_torch_fused",           # [A100] Exclusive ultra-fast optimizer
    optim="paged_adamw_8bit",          # [ T4 ] 8-bit paged saves massive VRAM (Comment out the A100 line above, uncomment this)

    save_strategy="epoch",
    logging_steps=10,
    report_to="tensorboard",

    # ================= Non-negotiable Universal Parameters =================
    learning_rate=1e-4,
    max_length=1536,
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    max_grad_norm=0.3
)

print("🤖 Assembling the full trainer...")

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("🚀 Starting training!")
trainer.train()

# ------------------------------------------
# ⏬ [ Resume Training Switch ] If Colab disconnected, comment out the trainer.train() above,
# and uncomment the line below, and click Run All. The trainer will automatically find the latest checkpoint!
# trainer.train(resume_from_checkpoint=True)

In [ ]:
# ==========================================
# 🔐 First Layer of Insurance: Save Original LoRA Adapter (Unmerged Version)
# ==========================================

# Define your exclusive storage path for LoRA
lora_save_path = "/content/drive/MyDrive/Kaggle_SVG/svg_lora_adapter_v5"

print(f"📦 Executing first layer of insurance: Saving LoRA adapter to {lora_save_path}...")

# Ensure the directory exists
if not os.path.exists(lora_save_path):
    os.makedirs(lora_save_path)

# Save LoRA weights and the corresponding tokenizer
model.save_pretrained(lora_save_path)
tokenizer.save_pretrained(lora_save_path)

print("✅ First layer of insurance locked in! LoRA adapter safely stored in Google Drive.")
print("🚀 Now you can safely and confidently execute the 'Model Merging' step below.")


In [ ]:
# ==========================================
# 🧬 Module 5: Model Merging
# ==========================================
# 1. Independent Dependency Check
!pip install -q transformers peft accelerate

# 2. Independent Import Block
import os
import gc
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 3. Independent Mount
drive.mount('/content/drive')

print("🧹 Cleaning VRAM...")
torch.cuda.empty_cache()
gc.collect()

# 4. Path Configuration
base_model_id = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
lora_adapter_path = "/content/drive/MyDrive/Kaggle_SVG/svg_lora_adapter_v5"
merged_dir = "/content/drive/MyDrive/Kaggle_SVG/merged_qwen_coder_1.5b_svg_final"

# 5. Execute Merge Logic
print("📥 Loading Base Model ...")

# ------------------------------------------
# [A100 Post-Training Merge] If you trained using A100 (bf16), please comment out the block below and uncomment this block:
# base_model_reload = AutoModelForCausalLM.from_pretrained(
#     base_model_id, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
# )
# ------------------------------------------
# ⏬ [T4 Post-Training Merge Switch] If you trained using T4 (fp16), please comment out the block above and uncomment the block below:
base_model_reload = AutoModelForCausalLM.from_pretrained(
    base_model_id, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)
# ------------------------------------------

print(f"📥 Reading LoRA Weights: {lora_adapter_path}")
model_to_merge = PeftModel.from_pretrained(base_model_reload, lora_adapter_path)

print("🧬 Writing adapter parameters into base model (Merge & Unload)...")
merged_model = model_to_merge.merge_and_unload()

# ⚡ [Universal Compatibility Layer] This is a foolproof step: whether you merged on A100 or T4,
# converting everything to FP16 for Kaggle's inference environment is absolutely safe. No T4 switch is needed here.
print("⚡ Converting precision -> FP16 (Adapting for Kaggle T4)...")
merged_model = merged_model.to(torch.float16)

print(f"💾 Saving final large model to: {merged_dir}")
tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
merged_model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print("✅ Merge Complete!")


In [ ]:
# ==========================================
# 🧪 Module 6: Evolution Gallery
# ==========================================

from google.colab import drive
drive.mount('/content/drive')
import re
import torch
import os
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display, HTML, SVG # Import native SVG renderer

# ==========================================
# 🧹 VRAM Physical Cleanup
# ==========================================

# 1. Completely delete variable references from the training phase
if 'trainer' in globals():
    del trainer
if 'model' in globals():
    del model

# 2. Force garbage collection
gc.collect()

# 3. Empty CUDA cache
torch.cuda.empty_cache()

# 4. Print current VRAM usage to confirm it is empty
print(f"✅ VRAM cleaned! Currently allocated VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# ------------------------------------------
# 🛠️ Part 1: Define Essential Functions
# ------------------------------------------

def show_svg(svg_code):
    print("🔍 Raw code generated by the model (Partial):")
    print(svg_code[:400] + "\n...")

    try:
        # If the model didn't write 'width', we force a 256x256 physical tag
        # This prevents Colab from stretching it to full screen
        if 'width=' not in svg_code:
            display_code = svg_code.replace('<svg', '<svg width="256" height="256"', 1)
        else:
            display_code = svg_code

        # Wrap it in a white background box so black lines are visible in Colab's dark mode
        display(HTML('<div style="background-color: white; display: inline-block; padding: 10px; border: 1px solid #ccc;">'))
        display(SVG(data=display_code))
        display(HTML('</div>'))
    except Exception as e:
        print(f"⚠️ Browser failed to render this SVG: {e}")

def generate_svg_evolved(model, tokenizer, prompt_text):
    # 🌟 Sync V5 System Prompt
    system_prompt = """You are an expert SVG code generator. Your task is to generate clean, strictly valid, and standalone SVG code based on the user's text description.

You MUST adhere to the following strict rules:
1. STRICT OUTPUT: Output ONLY the raw SVG code. No markdown formatting (no ```xml), no HTML wrappers, and no conversational text.
2. THE CANVAS RULE: Always use exactly <svg xmlns='[http://www.w3.org/2000/svg](http://www.w3.org/2000/svg)' viewBox='0 0 200 200' width='256' height='256'>.
3. THE ADAPTIVE KISS PRINCIPLE: For basic shapes, you MUST use primitives (<rect>, <circle>, <ellipse>, <line>, <polygon>, <polyline>). For complex/natural objects, use optimized <path> elements. ANTI-HALLUCINATION: NEVER invent non-existent tags like <triangle>, <square>, <star>, <curve>, <arc>, <background>, or <layer>. Use valid SVG alternatives (e.g., <polygon>, <rect>, <path>, <g>).
4. DRAWING ORDER: Render elements from back to front.
5. STYLE & COLOR: Use direct presentation attributes ONLY (e.g., fill='black').
6. SECURITY & VALIDITY: Ensure all tags are properly closed. Use single quotes for all attributes."""

    raw_prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(raw_prompt, return_tensors="pt").to("cuda")

    # Get the exact ChatML end token ID
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if im_end_id is None: im_end_id = 151645

    # Inject double EOS list to prevent infinite generation
    outputs = model.generate(
        **inputs,
        max_new_tokens=1536,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=[tokenizer.eos_token_id, im_end_id]
    )

    input_length = inputs.input_ids.shape[1]

    # Physically strip the <|im_end|> stop tag learned by the model
    generated_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=False)
    generated_text = generated_text.replace('<|im_end|>', '').strip()
    generated_text = generated_text.replace('```xml', '').replace('```', '')

    print("--- DEBUG: Raw text generated by the model ---")
    print(generated_text[:400] + "\n...")
    print("---------------------------------")

    # Kaggle-style regular expression
    try:
        match = re.search(r"<svg.*?>.*?</svg>", generated_text, flags=re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(0)
        else:
            print("❌ Regex scan failed: Could not find a complete <svg>...</svg> tag block!")
            return None
    except Exception as e:
        print(f"❌ Exception occurred during extraction: {e}")
        return None

# ------------------------------------------
# 🚀 Part 2: Execute Gallery Logic
# ------------------------------------------

# 1. Configure paths and parameters
base_path = '/content/drive/MyDrive/Kaggle_SVG/'
base_model_id = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# 🌟 Please replace with the actual checkpoints (Epoch checkpoints)
target_steps = [464, 928, 1392]

prompt = "A simple tree made of a green circle on top and a brown rectangle at the bottom as the trunk."

# 2. Load the pure base model and tokenizer
print("🔄 Loading pure Base Model and Tokenizer...")
# Clear VRAM to prevent Out-Of-Memory (OOM) errors
torch.cuda.empty_cache()
gc.collect()

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    token=None
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

print(f"\n==========================================\n🎨 [Evolution Gallery] Test Prompt: '{prompt}'\n==========================================")

# 3. Loop to load memories from different stages and draw
for step in target_steps:
    print(f"\n⏳ Loading Step {step} drawing skills LoRA...")
    lora_path = f"{base_path}results_full_train/checkpoint-{step}"

    # Load LoRA
    try:
        model_with_lora = PeftModel.from_pretrained(base_model, lora_path)
    except Exception as e:
        print(f"❌ Failed to load stage {step}, file not found (please check if the path is misspelled): {e}")
        continue

    print(f"🤖 Model is drawing the picture...")
    svg_result = generate_svg_evolved(model_with_lora, tokenizer, prompt)

    if svg_result:
        print(f"✅ Step {step} drawing complete! Result below:")
        show_svg(svg_result)

    # Unload current LoRA and clean VRAM to make room for the next loop
    del model_with_lora
    gc.collect()
    torch.cuda.empty_cache()

print("\n==========================================\n🎉 Evolution witnessed! You can compare and see which stage drew the best!")
